In [0]:
# ============================================================

# GOLD – dim_penalty

# Grain:

# (source_user_id, project, priority_level, penalty_code)

# ============================================================

from pyspark.sql import functions as F

from pyspark.sql.window import Window

# ---------------- CONFIG ----------------

SILVER_PENALTY_TABLE = "workspace.slainte_silver.penalty_silver_demo"

GOLD_DB = "workspace.slainte_gold"

DIM_PENALTY = f"{GOLD_DB}.dim_penalty"

# ---------------- SETUP ----------------

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

df_pen_silver = spark.table(SILVER_PENALTY_TABLE)

# ============================================================

# 1) Normalize base penalty data

# ============================================================

df_pen_base = (

    df_pen_silver

    .select(

        F.col("source_user_id"),

        F.col("project"),

        F.col("penalty_code"),

        F.col("priority_level"),

        F.col("penalty_type"),

        F.col("penalty_value"),

        F.col("penalty_unit"),

        F.col("conditions"),

        F.col("effective_from"),

        F.col("effective_to"),

        F.col("created_by"),

        F.col("created_at")

    )

    .filter(

        F.col("source_user_id").isNotNull() &

        F.col("project").isNotNull() &

        F.col("penalty_code").isNotNull()

    )

)

# ============================================================

# 2) Generate surrogate key

# ============================================================

penalty_window = Window.orderBy(

    "source_user_id",

    "project",

    "penalty_code",

    "priority_level",

    "effective_from"

)

df_dim_penalty = (

    df_pen_base

    .withColumn("penalty_id", F.row_number().over(penalty_window))

    .withColumn(

        "is_active",

        F.when(

            (F.col("effective_to").isNull()) |

            (F.col("effective_to") >= F.current_timestamp()),

            F.lit(1)

        ).otherwise(F.lit(0))

    )

    .select(

        "penalty_id",

        "source_user_id",

        "project",

        "penalty_code",

        "priority_level",

        "penalty_type",

        "penalty_value",

        "penalty_unit",

        "conditions",

        "effective_from",

        "effective_to",

        "is_active",

        "created_by",

        "created_at"

    )

)

# ============================================================

# 3) Write GOLD dimension

# ============================================================

df_dim_penalty.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_PENALTY)

# ============================================================

# 4) Preview

# ============================================================

display(

    spark.table(DIM_PENALTY)

        .orderBy("source_user_id", "project", "priority_level", "penalty_code")

)
 